<a href="https://colab.research.google.com/github/tellanad/AI-For-Beginners/blob/main/whisper_large_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Step 1: Install the latest version of the Transformers library
# This ensures we have access to the Whisper v3 model architecture.
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/openai/whisper-large-v3

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/openai/whisper-large-v3)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [10]:
# Step 2: Use the Transformers pipeline API
# This is the easiest way to perform ASR (Automatic Speech Recognition).
# It handles preprocessing, model execution, and decoding automatically.
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [11]:
# Step 3: Load the Processor and Model separately
# This method provides more flexibility, such as changing generation configs or using custom decoding loops.
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# The processor handles audio feature extraction and token decoding
processor = AutoProcessor.from_pretrained("openai/whisper-large-v3")
# The model itself performs the sequence-to-sequence generation
model = AutoModelForSpeechSeq2Seq.from_pretrained("openai/whisper-large-v3")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [27]:
# Step 4: Set up Environment for Remote Inference
# 1. Go to https://huggingface.co/settings/tokens
# 2. Copy your token.
# 3. Use the Colab 'Secrets' (key icon) on the left sidebar, name it 'HF_TOKEN', and paste your key.
# 4. Toggle the 'Notebook access' switch to ON.

from google.colab import userdata
try:
    import os
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('Token loaded successfully!')
except userdata.SecretNotFoundError:
    print('Please add your HF_TOKEN to the Colab Secrets panel.')

Token loaded successfully!


In [28]:
import kagglehub

In [22]:
path = kagglehub.dataset_download("kitkasemleepum/sampleflac")

100%|██████████| 11.8M/11.8M [00:01<00:00, 6.35MB/s]

Extracting files...


In [29]:
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/kitkasemleepum/sampleflac/versions/1


In [31]:
# Step 5: Initialize InferenceClient for free Serverless Inference
import os
import glob
from huggingface_hub import InferenceClient

# Initialize the client without 'provider="auto"' to default to HF Serverless (Free Tier)
client = InferenceClient(
    api_key=os.environ.get("HF_TOKEN"),
)

# Find a flac file in the kagglehub path
flac_files = glob.glob(os.path.join(path, "**/*.flac"), recursive=True)

if not flac_files:
    print("No .flac files found in the dataset path.")
else:
    audio_to_process = flac_files[0]
    print(f"Processing: {audio_to_process}")

    try:
        # Perform ASR using Hugging Face Serverless Inference
        # We specify the model directly to avoid the paid 'router' providers
        output = client.automatic_speech_recognition(
            audio_to_process,
            model="openai/whisper-large-v3"
        )
        print("\nTranscription:", output)
    except Exception as e:
        print(f"\nAn error occurred: {e}")
        print("If the error persists, ensure your HF_TOKEN is valid and has 'Read' permissions.")

Processing: /root/.cache/kagglehub/datasets/kitkasemleepum/sampleflac/versions/1/sample1.flac

An error occurred: Client error '402 Payment Required' for url 'https://router.huggingface.co/fal-ai/fal-ai/whisper' (Request ID: Root=1-69921699-677fe1911ddd09eb53415d28;ec6a6b90-8e0f-4717-a9b3-cb22a3001415)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Pre-paid credits are required to use provider fal-ai. Add pre-paid credits to your account to start using fal-ai.
If the error persists, ensure your HF_TOKEN is valid and has 'Read' permissions.
